# REMD MBAR Free Energy Analysis

Based on MBAR_Parameter_Analysis-v2.ipynb

This notebook performs free energy surface (FES) analysis using the MBAR method.

In [ ]:
"""
REMD MBAR FES Analysis Notebook
Author: Song Yang
Date: 2025
"""

import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from pathlib import Path

# Import FESAnalyzer from CTGoMartini
from ctgomartini.analysis.remd_mbar import FESAnalyzer

# Set plotting style with Arial font
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False

print('Imports successful!')

## 1. Setup and ConfigurationDefine input files and analysis parameters.

In [ ]:
# Configuration
OUTPUT_FILE = 'output.nc'
CV_FILE = './dRMStraj_nc_StateA.dat'  # or StateB
INTERVAL = 5  # CV data interval

# Analysis parameters
ANALYSIS_PARAMS = {
    'g': 1,                    # Statistical inefficiency
    'length_ratio': 1.0,       # Use full trajectory
    'start_point': None,       # Start from specific frame
    'start_ratio': 0.2,        # Skip first 20% for equilibration
    'selected_state': 5        # Thermodynamic state to analyze
}

# dRMS bounds for barrier search (from v2 notebook)
LEFT_BOUND = 3
RIGHT_BOUND = 8

# Output directories
os.makedirs('AnalysisParameter', exist_ok=True)
os.makedirs('FitEnergy', exist_ok=True)

print(f'Output file: {OUTPUT_FILE}')
print(f'CV file: {CV_FILE}')
print(f'Bounds: [{LEFT_BOUND}, {RIGHT_BOUND}]')

## 2. Initialize AnalyzerLoad simulation data and initialize FES analyzer.

In [ ]:
# Initialize analyzer
analyzer = FESAnalyzer(OUTPUT_FILE, CV_FILE, interval=INTERVAL)

print(f'Number of states: {analyzer.n_states}')
print(f'Temperatures: {analyzer.temperatures_k}')
print(f'CV shape: {analyzer.cv_values_replica.shape}')

## 3. Single State AnalysisAnalyze free energy surface for a single state.

In [ ]:
# Initialize FES
analyzer.initialize_fes(
    g=ANALYSIS_PARAMS['g'],
    length_ratio=ANALYSIS_PARAMS['length_ratio'],
    start_ratio=ANALYSIS_PARAMS['start_ratio']
)

# Analyze single state
results = analyzer.analyze_onestate(
    selected_state=ANALYSIS_PARAMS['selected_state'],
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)

print('Metrics:')
for key, value in results['metrics'].items():
    print(f'  {key}: {value:.4f}')

In [ ]:
# Plot PMF
fig, ax = plt.subplots(figsize=(10, 6))

cv_values = results['cv_values']
pmf = results['pmf']
pmf_uncertainty = results['pmf_uncertainty']
metrics = results['metrics']

pmf_normalized = pmf - pmf.min()

ax.plot(cv_values, pmf_normalized, 'b-', linewidth=2, label=f"State {ANALYSIS_PARAMS['selected_state']}")
ax.fill_between(cv_values, 
                pmf_normalized - pmf_uncertainty, 
                pmf_normalized + pmf_uncertainty, 
                alpha=0.3, color='b')

ax.axvline(metrics['barrier_pos'], color='r', linestyle='--', label=f"Barrier: {metrics['barrier_pos']:.2f}")
ax.axvline(metrics['basin1_pos'], color='g', linestyle=':', alpha=0.7)
ax.axvline(metrics['basin2_pos'], color='g', linestyle=':', alpha=0.7)

ax.set_xlabel('dRMS (nm)', fontsize=14)
ax.set_ylabel('Free Energy (kJ/mol)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Barrier: {metrics['barrier']:.2f} kJ/mol, Keq: {metrics['keq']:.4f}")

## 4. Parameter Sweep AnalysisCheck convergence and sensitivity.

In [ ]:
# Start ratio sweep
print('Start ratio sweep...')
start_ratio_results = analyzer.parameter_sweep(
    'start_ratio', 
    [0, 0.1, 0.2, 0.3, 0.4, 0.5],
    default_params=ANALYSIS_PARAMS,
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)
analyzer.save_results(start_ratio_results, 'AnalysisParameter/start_ratio_results.pkl')

print('Statistical inefficiency sweep...')
g_values_results = analyzer.parameter_sweep(
    'g', 
    [1, 5, 10, 20, 50],
    default_params=ANALYSIS_PARAMS,
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)
analyzer.save_results(g_values_results, 'AnalysisParameter/g_values_results.pkl')

print('Length ratio sweep...')
length_ratio_results = analyzer.parameter_sweep(
    'length_ratio', 
    [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    default_params=ANALYSIS_PARAMS,
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)
analyzer.save_results(length_ratio_results, 'AnalysisParameter/length_ratio_results.pkl')

print('State sweep...')
state_results = analyzer.parameter_sweep(
    'selected_state', 
    list(range(analyzer.n_states)),
    default_params=ANALYSIS_PARAMS,
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)
analyzer.save_results(state_results, 'AnalysisParameter/selected_state_results.pkl')

print('All sweeps completed!')

In [ ]:
# Plot parameter sweeps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Start ratio
ratios = list(start_ratio_results.keys())
barriers = [start_ratio_results[r]['metrics']['barrier'] for r in ratios]
keqs = [start_ratio_results[r]['metrics']['keq'] for r in ratios]
axes[0,0].plot(ratios, barriers, 'o-', linewidth=2)
axes[0,0].set_xlabel('Start Ratio', fontsize=12)
axes[0,0].set_ylabel('Barrier (kJ/mol)', fontsize=12)
axes[0,0].set_title('Barrier vs Equilibration', fontsize=13)
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(ratios, keqs, 's-', linewidth=2, color='orange')
axes[0,1].set_xlabel('Start Ratio', fontsize=12)
axes[0,1].set_ylabel('Keq', fontsize=12)
axes[0,1].set_title('Keq vs Equilibration', fontsize=13)
axes[0,1].grid(True, alpha=0.3)

# Length ratio
lengths = list(length_ratio_results.keys())
barriers = [length_ratio_results[l]['metrics']['barrier'] for l in lengths]
keqs = [length_ratio_results[l]['metrics']['keq'] for l in lengths]
axes[1,0].plot(lengths, barriers, 'o-', linewidth=2, color='green')
axes[1,0].set_xlabel('Length Ratio', fontsize=12)
axes[1,0].set_ylabel('Barrier (kJ/mol)', fontsize=12)
axes[1,0].set_title('Barrier vs Trajectory Length', fontsize=13)
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(lengths, keqs, 's-', linewidth=2, color='red')
axes[1,1].set_xlabel('Length Ratio', fontsize=12)
axes[1,1].set_ylabel('Keq', fontsize=12)
axes[1,1].set_title('Keq vs Trajectory Length', fontsize=13)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Mixed-State Analysis (EXP and HAM)Using parameters from v2 notebook.

In [ ]:
# EXP Mixing
exp_params = {
    'beta': 1/300,
    'C1': -300,
    'C2': 0
}

exp_results = analyzer.analyze_one_mixing_parameters(
    mixing_parameters=exp_params,
    temperature=310,
    method='EXP',
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)

exp_results['mixing_parameters'] = exp_params
exp_results['temperature'] = 310
exp_results['method'] = 'EXP'

analyzer.save_results(exp_results, 'FitEnergy/FreeEnergy_EXP.pkl')
print('EXP:', exp_results['metrics'])

In [ ]:
# HAM Mixing
ham_params = {
    'delta': 350,
    'C1': -348,
    'C2': 0
}

ham_results = analyzer.analyze_one_mixing_parameters(
    mixing_parameters=ham_params,
    temperature=310,
    method='HAM',
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)

ham_results['mixing_parameters'] = ham_params
ham_results['temperature'] = 310
ham_results['method'] = 'HAM'

analyzer.save_results(ham_results, 'FitEnergy/FreeEnergy_HAM.pkl')
print('HAM:', ham_results['metrics'])

In [ ]:
# Compare EXP and HAM
fig, ax = plt.subplots(figsize=(10, 6))

exp_pmf = exp_results['pmf'] - exp_results['pmf'].min()
ax.plot(exp_results['cv_values'], exp_pmf, 'b-', linewidth=2.5, label='EXP')
ax.fill_between(exp_results['cv_values'],
                exp_pmf - exp_results['pmf_uncertainty'],
                exp_pmf + exp_results['pmf_uncertainty'],
                alpha=0.2, color='b')

ham_pmf = ham_results['pmf'] - ham_results['pmf'].min()
ax.plot(ham_results['cv_values'], ham_pmf, 'r-', linewidth=2.5, label='HAM')
ax.fill_between(ham_results['cv_values'],
                ham_pmf - ham_results['pmf_uncertainty'],
                ham_pmf + ham_results['pmf_uncertainty'],
                alpha=0.2, color='r')

ax.set_xlabel('dRMS (nm)', fontsize=14)
ax.set_ylabel('Free Energy (kJ/mol)', fontsize=14)
ax.set_title('EXP vs HAM Mixing', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nEXP: Barrier={exp_results['metrics']['barrier']:.2f}, Keq={exp_results['metrics']['keq']:.4f}")
print(f"HAM: Barrier={ham_results['metrics']['barrier']:.2f}, Keq={ham_results['metrics']['keq']:.4f}")

## 6. SummaryGenerate summary report.

In [ ]:
# Summary report
print('='*60)
print('MBAR ANALYSIS SUMMARY')
print('='*60)
print(f"\nSystem: {OUTPUT_FILE}")
print(f"CV file: {CV_FILE}")
print(f"Interval: {INTERVAL}")
print(f"Bounds: [{LEFT_BOUND}, {RIGHT_BOUND}]")
print(f"\nAnalysis State: {ANALYSIS_PARAMS['selected_state']}")
print(f"Start Ratio: {ANALYSIS_PARAMS['start_ratio']}")
print(f"g: {ANALYSIS_PARAMS['g']}")

print('\nSingle State Results:')
print(f"  Barrier: {results['metrics']['barrier']:.2f} kJ/mol")
print(f"  Keq: {results['metrics']['keq']:.4f}")

print('\nEXP Mixing Results:')
print(f"  Barrier: {exp_results['metrics']['barrier']:.2f} kJ/mol")
print(f"  Keq: {exp_results['metrics']['keq']:.4f}")

print('\nHAM Mixing Results:')
print(f"  Barrier: {ham_results['metrics']['barrier']:.2f} kJ/mol")
print(f"  Keq: {ham_results['metrics']['keq']:.4f}")

print('\nSaved Files:')
print('  AnalysisParameter/start_ratio_results.pkl')
print('  AnalysisParameter/g_values_results.pkl')
print('  AnalysisParameter/length_ratio_results.pkl')
print('  AnalysisParameter/selected_state_results.pkl')
print('  FitEnergy/FreeEnergy_EXP.pkl')
print('  FitEnergy/FreeEnergy_HAM.pkl')
print('='*60)